# 21 cm × Galaxy Cross-Correlation — Part 3: Power Spectra & SNR
## HERA × Euclid (Lightcone)

This notebook loads the simulation outputs produced by `run_simulation.py`
and performs all post-simulation calculations:

1. **2D cylindrical power spectra** $P_{21}$, $P_{\rm gal}$, $P_{21\times{\rm gal}}$
2. **Foreground wedge geometry** — horizon and HERA FoV wedge lines
3. **Photo-$z$ damping** — Euclid photometric-redshift smearing along LOS
4. **SNR map** — per-mode and cumulative detection significance

**Prerequisites:** run `run_simulation.py` (or `sbatch submit_job.sh`) first
to generate `outputs/lightcone_data.h5`.

**References:**
- Davies, Mesinger & Murray (2025) — [arXiv:2504.17254](https://arxiv.org/abs/2504.17254)
- Gagnon-Hartman, Davies & Mesinger (2025) — [arXiv:2502.20447](https://arxiv.org/abs/2502.20447)
- La Plante et al. (2023) — [arXiv:2205.09770](https://arxiv.org/abs/2205.09770)
- Euclid Collaboration (2022) — [arXiv:2108.01201](https://arxiv.org/abs/2108.01201)


## Imports and setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.ndimage import generic_filter
import h5py
import warnings

warnings.filterwarnings('ignore')

# ── Matplotlib defaults ──────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":      300,
    "font.size":       12,
    "axes.labelsize":  13,
    "legend.fontsize": 10,
    "axes.grid":       False,
})


## Configuration — set path to simulation output

Edit `OUTPUT_FILE` if you saved the HDF5 to a different location.


In [ ]:
# ── Path to simulation output (produced by run_simulation.py) ────────────────
OUTPUT_FILE = "../outputs/lightcone_data.h5"

# ── Load all fields and metadata from HDF5 ───────────────────────────────────
with h5py.File(OUTPUT_FILE, 'r') as f:

    # Simulation fields
    brightness_temp_field = f['brightness_temp_field'][:]
    neutral_fraction      = f['neutral_fraction'][:]
    galaxy_overdensity    = f['galaxy_overdensity'][:]

    # LOS geometry
    lc_redshifts = f['lc_redshifts'][:]

    # Scalar metadata
    HII_DIM             = int(f.attrs['HII_DIM'])
    BOX_LEN             = float(f.attrs['BOX_LEN'])
    N_z                 = int(f.attrs['N_z'])
    L_los               = float(f.attrs['L_los'])
    cell_size           = float(f.attrs['cell_size'])
    z_min               = float(f.attrs['z_min'])
    z_max               = float(f.attrs['z_max'])
    z_obs               = float(f.attrs['z_obs'])
    galaxy_bias         = float(f.attrs['galaxy_bias'])
    mean_galaxy_density = float(f.attrs['mean_galaxy_density'])
    photoz_uncertainty  = float(f.attrs['photoz_uncertainty'])
    M_UV_limit          = float(f.attrs['M_UV_limit'])
    OMEGA_M_0           = float(f.attrs['OMEGA_M_0'])
    HUBBLE_CONSTANT     = float(f.attrs['HUBBLE_CONSTANT'])
    SPEED_OF_LIGHT_KMS  = float(f.attrs['SPEED_OF_LIGHT_KMS'])
    SPEED_OF_LIGHT_MPS  = float(f.attrs['SPEED_OF_LIGHT_MPS'])
    F_21_MHZ            = float(f.attrs['F_21_MHZ'])
    F_21_HZ             = float(f.attrs['F_21_HZ'])
    HERA_DISH_DIAMETER  = float(f.attrs['HERA_DISH_DIAMETER'])
    integration_time    = float(f.attrs['integration_time'])
    bandwidth           = float(f.attrs['bandwidth'])
    wedge_buffer        = float(f.attrs['wedge_buffer'])
    n_bins_perp         = int(f.attrs['n_bins_perp'])
    n_bins_parallel     = int(f.attrs['n_bins_parallel'])

# ── Hubble parameter (needed for wedge/noise calculations) ───────────────────
def hubble_parameter(z):
    """H(z) for flat ΛCDM  [km s⁻¹ Mpc⁻¹]."""
    return HUBBLE_CONSTANT * np.sqrt(OMEGA_M_0 * (1 + z)**3 + (1 - OMEGA_M_0))


print(f'Loaded: brightness_temp_field {brightness_temp_field.shape}')
print(f'        neutral_fraction       {neutral_fraction.shape}')
print(f'        galaxy_overdensity     {galaxy_overdensity.shape}')
print(f'        z = {z_min} -> {z_max},  z_obs = {z_obs}')
print(f'        galaxy_bias = {galaxy_bias:.3f}')


## 1  Compute the 2D cylindrical power spectra

The lightcone box is non-cubic: $(HII\_DIM, HII\_DIM, N_z)$ with transverse
size $L_\perp = BOX\_LEN$ and LOS size $L_{\rm LOS}$. The function below
handles this by using separate cell sizes and fundamental modes for the
transverse and LOS directions.

We compute:
- $P_{21}(k_\perp, k_\parallel)$ — 21 cm auto-power
- $P_{\rm gal}(k_\perp, k_\parallel)$ — galaxy auto-power
- $P_{21 \times \rm gal}(k_\perp, k_\parallel)$ — cross-power (expected to be **negative** on large scales)


In [ ]:
def compute_cylindrical_cross_power(
    field_a,
    field_b,
    box_len_perp,
    box_len_los,
    n_bins_perp=20,
    n_bins_parallel=20,
):
    """
    Compute the 2D cylindrical cross-power spectrum P(k_perp, k_parallel)
    for a non-cubic (lightcone) box.

    Parameters
    ----------
    field_a, field_b : ndarray, shape (N, N, N_z)
        Real-space 3D fields. Transverse dimensions are equal (N × N);
        the LOS dimension N_z may differ.
    box_len_perp : float
        Comoving side length of the transverse box  [Mpc].
    box_len_los : float
        Comoving length of the LOS axis  [Mpc].
    n_bins_perp, n_bins_parallel : int
        Number of log-spaced bins along k_perp and k_parallel.

    Returns
    -------
    k_perp_centres     : 1D array  [Mpc⁻¹]
    k_parallel_centres : 1D array  [Mpc⁻¹]
    power_2d           : 2D array, shape (n_bins_perp, n_bins_parallel)
    mode_counts        : 2D array, shape (n_bins_perp, n_bins_parallel)
    """
    N   = field_a.shape[0]   # transverse cells
    N_z = field_a.shape[2]   # LOS cells

    dx     = box_len_perp / N    # transverse cell size  [Mpc]
    dz     = box_len_los  / N_z  # LOS cell size  [Mpc]
    volume = box_len_perp**2 * box_len_los

    # ── Fourier transforms ────────────────────────────────────────────
    ft_factor = dx * dx * dz
    fourier_a = np.fft.fftn(field_a) * ft_factor
    fourier_b = np.fft.fftn(field_b) * ft_factor
    power_3d  = (fourier_a * np.conj(fourier_b)).real / volume

    # ── Wavenumber grids ──────────────────────────────────────────────
    kx = np.fft.fftfreq(N,   d=dx) * 2 * np.pi
    ky = np.fft.fftfreq(N,   d=dx) * 2 * np.pi
    kz = np.fft.fftfreq(N_z, d=dz) * 2 * np.pi
    KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')

    k_perp_3d     = np.sqrt(KX**2 + KY**2)
    k_parallel_3d = np.abs(KZ)

    # ── Bin edges (log-spaced, starting below first discrete mode) ────
    dk_perp = 2 * np.pi / box_len_perp
    dk_par  = 2 * np.pi / box_len_los

    k_max_perp = np.sqrt(2) * np.abs(kx).max() * 1.05
    k_max_par  = np.abs(kz).max() * 1.05

    k_perp_edges = np.logspace(
        np.log10(0.5 * dk_perp), np.log10(k_max_perp), n_bins_perp + 1
    )
    k_par_edges = np.logspace(
        np.log10(0.5 * dk_par), np.log10(k_max_par), n_bins_parallel + 1
    )

    # ── Bin the 3D power into 2D ──────────────────────────────────────
    power_2d    = np.zeros((n_bins_perp, n_bins_parallel))
    mode_counts = np.zeros_like(power_2d)

    bin_perp = np.digitize(k_perp_3d.ravel(), k_perp_edges)    - 1
    bin_par  = np.digitize(k_parallel_3d.ravel(), k_par_edges) - 1
    power_flat = power_3d.ravel()

    inside = (
        (bin_perp >= 0) & (bin_perp < n_bins_perp) &
        (bin_par  >= 0) & (bin_par  < n_bins_parallel)
    )
    np.add.at(power_2d,    (bin_perp[inside], bin_par[inside]), power_flat[inside])
    np.add.at(mode_counts, (bin_perp[inside], bin_par[inside]), 1)

    power_2d = np.divide(
        power_2d, mode_counts,
        where=mode_counts > 0,
        out=np.full_like(power_2d, np.nan),
    )

    k_perp_centres = np.sqrt(k_perp_edges[:-1] * k_perp_edges[1:])
    k_par_centres  = np.sqrt(k_par_edges[:-1]  * k_par_edges[1:])

    return k_perp_centres, k_par_centres, power_2d, mode_counts


# ======================================================================
#  Compute all three 2D power spectra
# ======================================================================
T21_fluctuations    = brightness_temp_field - brightness_temp_field.mean()
galaxy_fluctuations = galaxy_overdensity

print('Computing 2D cylindrical power spectra …')

k_perp, k_parallel, P_21cm_auto, mode_counts = compute_cylindrical_cross_power(
    T21_fluctuations, T21_fluctuations,
    BOX_LEN, L_los, n_bins_perp, n_bins_parallel,
)
_, _, P_galaxy_auto, _ = compute_cylindrical_cross_power(
    galaxy_fluctuations, galaxy_fluctuations,
    BOX_LEN, L_los, n_bins_perp, n_bins_parallel,
)
_, _, P_cross, _ = compute_cylindrical_cross_power(
    T21_fluctuations, galaxy_fluctuations,
    BOX_LEN, L_los, n_bins_perp, n_bins_parallel,
)

large_scale_mean = np.nanmean(P_cross[:5, :5])
n_empty          = np.sum(mode_counts == 0)
print(f'k_perp     range : [{k_perp.min():.4f},  {k_perp.max():.3f}] Mpc⁻¹')
print(f'k_parallel range : [{k_parallel.min():.4f},  {k_parallel.max():.3f}] Mpc⁻¹')
print(f'Empty bins       : {n_empty} / {mode_counts.size}')
print(
    f'Large-scale cross-spectrum sign: '
    f'{"NEGATIVE (anti-correlated) ✓" if large_scale_mean < 0 else "POSITIVE"}'
)


## 2  Plot the 2D cylindrical power spectra

Three panels showing $P_{21}$, $P_{\rm gal}$, and the signed cross-power
$P_{21\times\rm gal}$ in $(k_\perp,\, k_\parallel)$ space, with the
foreground horizon and HERA primary-beam wedge overlaid.


In [ ]:
def fill_nan_nearest(arr):
    """Replace NaN pixels with the nearest non-NaN neighbour (display only)."""
    mask = np.isnan(arr)
    if not mask.any():
        return arr
    filled = arr.copy()
    for _ in range(max(arr.shape)):
        if not np.isnan(filled).any():
            break
        kernel_filled = generic_filter(filled, np.nanmean, size=3, mode='nearest')
        filled = np.where(np.isnan(filled), kernel_filled, filled)
    return filled


# ======================================================================
#  Foreground wedge geometry  (Thyagarajan+2015; La Plante+2023, Eq. 10)
#  Evaluated at the reference redshift z_obs
# ======================================================================
Hz_obs = hubble_parameter(z_obs)   # H(z_obs)  [km s⁻¹ Mpc⁻¹]

comoving_distance_obs, _ = quad(
    lambda z_: SPEED_OF_LIGHT_KMS / hubble_parameter(z_), 0, z_obs
)   # D_c(z_obs)  [Mpc]

lambda_obs = SPEED_OF_LIGHT_MPS * (1 + z_obs) / F_21_HZ   # observed wavelength  [m]

# Horizon wedge slope
horizon_slope = (
    lambda_obs * comoving_distance_obs * F_21_HZ * Hz_obs
    / (SPEED_OF_LIGHT_MPS * SPEED_OF_LIGHT_KMS * (1 + z_obs)**2)
)

# HERA primary-beam wedge slope
theta_fov      = lambda_obs / HERA_DISH_DIAMETER
fov_wedge_slope = np.sin(theta_fov) * horizon_slope

k_perp_line   = np.logspace(np.log10(k_perp.min()), np.log10(k_perp.max()), 200)
k_par_horizon = k_perp_line * horizon_slope
k_par_fov     = k_perp_line * fov_wedge_slope

print(f'Reference z_obs        : {z_obs}')
print(f'Observed wavelength    : λ = {lambda_obs:.3f} m')
print(f'Comoving distance      : D_c = {comoving_distance_obs:.0f} Mpc')
print(f'Horizon wedge slope    : m = {horizon_slope:.3f}')
print(f'HERA beam half-angle   : θ = {np.degrees(theta_fov):.1f}°')
print(f'HERA FoV wedge slope   : m_FoV = {fov_wedge_slope:.3f}')

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: 21 cm auto-power
P21_display = fill_nan_nearest(np.log10(np.abs(P_21cm_auto.T)))
im0 = axes[0].pcolormesh(
    k_perp, k_parallel, P21_display, cmap='viridis', shading='auto'
)
fig.colorbar(im0, ax=axes[0], label=r"$\log_{10} |P_{21}|$  [mK² Mpc³]")
axes[0].plot(k_perp_line, k_par_horizon, "w-",  lw=1.2, alpha=0.7, label="Horizon")
axes[0].plot(k_perp_line, k_par_fov,     "w--", lw=1.2, alpha=0.8, label="HERA FoV wedge")
axes[0].legend(loc='upper left', fontsize=8)
axes[0].set_title(r"$P_{21}(k_\perp,\, k_\parallel)$")

# Panel 2: galaxy auto-power
Pgal_display = fill_nan_nearest(np.log10(np.abs(P_galaxy_auto.T)))
im1 = axes[1].pcolormesh(
    k_perp, k_parallel, Pgal_display, cmap='plasma', shading='auto'
)
fig.colorbar(im1, ax=axes[1], label=r"$\log_{10} |P_{\rm gal}|$  [Mpc³]")
axes[1].plot(k_perp_line, k_par_horizon, "w-",  lw=1.2, alpha=0.7)
axes[1].plot(k_perp_line, k_par_fov,     "w--", lw=1.2, alpha=0.8)
axes[1].set_title(r"$P_{\rm gal}(k_\perp,\, k_\parallel)$")

# Panel 3: signed cross-power
cross_amp      = np.abs(P_cross.T)
cross_sgn      = np.sign(P_cross.T)
signed_log_raw = np.where(cross_amp > 0, np.log10(cross_amp) * cross_sgn, 0.0)
signed_log     = fill_nan_nearest(signed_log_raw)
clim           = np.nanpercentile(np.abs(signed_log[signed_log != 0]), 95)

im2 = axes[2].pcolormesh(
    k_perp, k_parallel, signed_log,
    cmap='RdBu_r', shading='auto', vmin=-clim, vmax=clim,
)
fig.colorbar(im2, ax=axes[2],
             label=r"sign $\times$ $\log_{10} |P_{21 \times \rm gal}|$")
axes[2].plot(k_perp_line, k_par_horizon, "k-",  lw=1.2, alpha=0.7)
axes[2].plot(k_perp_line, k_par_fov,     "k--", lw=1.2, alpha=0.8)
axes[2].set_title(r"$P_{21 \times \rm gal}$  (signed)")

mean_xHI = np.mean(neutral_fraction)
for ax in axes:
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r"$k_\perp$  [Mpc$^{-1}$]")
    ax.set_ylabel(r"$k_\parallel$  [Mpc$^{-1}$]")
    ax.set_xlim(k_perp[0], k_perp[-1])
    ax.set_ylim(k_parallel[0], k_parallel[-2])

plt.suptitle(
    rf"2D cylindrical power spectra — lightcone $z = {z_min}$–${z_max}$   "
    rf"($\langle x_{{\rm HI}} \rangle = {mean_xHI:.2f}$,  "
    rf"Euclid $M_{{\rm UV}} < {M_UV_limit}$)",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()


## 3  Apply photo-$z$ damping and foreground wedge

Same procedure as the coeval notebook, evaluated at the reference redshift
$z_{\rm obs}$:

1. **Photo-$z$ damping** — Euclid photometric errors smear the galaxy field
   along the LOS, suppressing $P_{\rm gal}$ by $W^2$ and
   $P_{21\times\rm gal}$ by $W$.
2. **Foreground wedge** — modes with
   $k_\parallel < k_\perp \cdot \chi H / [c(1+z)]$ are excised.


In [ ]:
# ── Photo-z radial smearing  σ_r = c σ_z / H(z_obs) ─────────────────────
Hz_obs          = hubble_parameter(z_obs)
radial_smearing = SPEED_OF_LIGHT_KMS * photoz_uncertainty / Hz_obs   # [Mpc]

# Broadcast k_parallel over the 2D power spectrum grid
K_PAR_BROADCAST = k_parallel[np.newaxis, :]   # (1, n_bins_par)
photoz_kernel   = np.exp(-0.5 * K_PAR_BROADCAST**2 * radial_smearing**2)

# Apply damping (one factor of W for cross, two for galaxy auto)
P_cross_observed  = P_cross       * photoz_kernel
P_galaxy_observed = P_galaxy_auto * photoz_kernel**2

# ── Foreground wedge mask ──────────────────────────────────────────────────
comoving_dist_obs, _ = quad(
    lambda z_: SPEED_OF_LIGHT_KMS / hubble_parameter(z_), 0, z_obs
)
wedge_slope = comoving_dist_obs * Hz_obs / (SPEED_OF_LIGHT_KMS * (1 + z_obs))

K_PERP_GRID, K_PAR_GRID = np.meshgrid(k_perp, k_parallel, indexing='ij')
outside_wedge = K_PAR_GRID > (K_PERP_GRID * wedge_slope + wedge_buffer)

print(f'Photo-z smearing : σ_r = {radial_smearing:.1f} Mpc')
print(f'Wedge slope      : {wedge_slope:.3f}')
print(f'Modes outside wedge : {outside_wedge.sum() / outside_wedge.size:.1%}')


## 4  Cross-correlation SNR map

In [ ]:
# ── 21 cm thermal noise (simplified HERA-like estimate) ──────────────────
observed_frequency = 1.42e9 / (1 + z_obs)   # Hz
system_temperature = (
    100 + 60 * (300e6 / observed_frequency)**2.55
) * 1e3   # mK

P_noise_21cm = (
    system_temperature**2 * 1e3 / (integration_time * bandwidth)
    * np.ones_like(P_cross)
)

# ── Galaxy shot noise ──────────────────────────────────────────────────────
P_noise_galaxy = 1.0 / mean_galaxy_density

# ── Per-mode uncertainty (La Plante et al. 2023, Eqs. 15–17) ──────────────
sigma_21cm   = np.abs(P_21cm_auto)       + P_noise_21cm
sigma_galaxy = np.abs(P_galaxy_observed) + P_noise_galaxy
sigma_cross  = np.sqrt(
    0.5 * (P_cross_observed**2 + sigma_21cm * sigma_galaxy)
)

# ── SNR maps ───────────────────────────────────────────────────────────────
SNR_per_mode      = np.abs(P_cross_observed) / sigma_cross
SNR_outside_wedge = np.where(outside_wedge, SNR_per_mode, np.nan)

# ── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Wedge and horizon lines for overlay
k_perp_line    = np.logspace(np.log10(k_perp.min()), np.log10(k_perp.max()), 200)
lambda_obs_    = SPEED_OF_LIGHT_KMS * 1e3 * (1 + z_obs) / F_21_HZ
horizon_slope_ = (
    lambda_obs_ * comoving_dist_obs * F_21_HZ * Hz_obs
    / (SPEED_OF_LIGHT_KMS * 1e3 * SPEED_OF_LIGHT_KMS * (1 + z_obs)**2)
)
theta_fov_  = lambda_obs_ / HERA_DISH_DIAMETER
fov_slope_  = np.sin(theta_fov_) * horizon_slope_

k_par_horizon_ = k_perp_line * horizon_slope_
k_par_fov_     = k_perp_line * fov_slope_

# Panel 1: per-mode SNR
snr_display = fill_nan_nearest(np.log10(SNR_per_mode.T))
im_snr = axes[0].pcolormesh(
    k_perp, k_parallel, snr_display,
    cmap='magma', shading='auto', vmin=-3, vmax=0,
)
fig.colorbar(im_snr, ax=axes[0], label=r"$\log_{10}$ SNR per mode")
axes[0].plot(k_perp_line, k_par_horizon_, "w-",  lw=1.2, alpha=0.7, label="Horizon")
axes[0].plot(k_perp_line, k_par_fov_,     "w--", lw=1.2, alpha=0.8, label="HERA FoV wedge")
axes[0].legend(loc='upper left', fontsize=8)
axes[0].set_title('Per-mode SNR')

# Panel 2: observed cross-power
cross_amp_  = np.abs(P_cross_observed.T)
cross_sgn_  = np.sign(P_cross_observed.T)
signed_raw_ = np.where(cross_amp_ > 0, np.log10(cross_amp_) * cross_sgn_, 0.0)
signed_log_ = fill_nan_nearest(signed_raw_)
clim_       = np.nanpercentile(np.abs(signed_log_[signed_log_ != 0]), 95)

im_cross = axes[1].pcolormesh(
    k_perp, k_parallel, signed_log_,
    cmap='RdBu_r', shading='auto', vmin=-clim_, vmax=clim_,
)
fig.colorbar(
    im_cross, ax=axes[1],
    label=r"sign $\times$ $\log_{10}|P_{21 \times \rm gal}^{\rm obs}|$",
)
axes[1].plot(k_perp_line, k_par_horizon_, "k-",  lw=1.2, alpha=0.7)
axes[1].plot(k_perp_line, k_par_fov_,     "k--", lw=1.2, alpha=0.8)
axes[1].set_title(r"Observed $P_{21 \times \rm gal}$  (photo-$z$ damped)")

for ax in axes:
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r"$k_\perp$  [Mpc$^{-1}$]")
    ax.set_ylabel(r"$k_\parallel$  [Mpc$^{-1}$]")
    ax.set_xlim(k_perp[0], k_perp[-1])
    ax.set_ylim(k_parallel[0], k_parallel[-2])

plt.suptitle(
    rf"HERA $\times$ Euclid — lightcone $z = {z_min}$–${z_max}$   "
    rf"($\sigma_z = {photoz_uncertainty}$)",
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

# ── Total detection significance ───────────────────────────────────────────
valid_snr = SNR_per_mode[outside_wedge]
total_snr = np.sqrt(np.nansum(valid_snr**2))

print(f'\nTotal cross-correlation SNR (outside wedge): {total_snr:.1f} σ')
print(f'Detection (> 5σ) ?  {"YES" if total_snr > 5 else "NO"}')


## 5  Summary

This notebook demonstrates the post-simulation analysis for a
HERA × Euclid 21 cm–galaxy cross-correlation using a **21cmFASTv4 lightcone**:

1. **21cmFASTv4** generates a self-consistent lightcone via `RectilinearLightconer`
   + `run_lightcone`, covering $z = z_{\rm min}$–$z_{\rm max}$ in a single run
2. The **non-cubic box** $(N_\perp \times N_\perp \times N_z)$ is handled
   throughout, with separate cell sizes for the transverse and LOS directions
3. A **Euclid-like** galaxy field is extracted from the lightcone `halo_sfr`
4. The **2D cross-power spectrum** $P_{21 \times \rm gal}(k_\perp, k_\parallel)$
   is computed and shows the expected large-scale anti-correlation
5. **Photo-$z$ damping** and **foreground wedge** excision are applied at
   $z_{\rm obs}$
6. The **SNR map** identifies which Fourier modes drive the detection

### Key differences vs. the coeval version

| Feature | Coeval | Lightcone |
|---|---|---|
| 21cmFAST function | `run_coeval` | `run_lightcone` |
| Box shape | $(N, N, N)$ | $(N, N, N_z)$ |
| Redshift | single snapshot | continuous $z_{\rm min}$–$z_{\rm max}$ |
| LOS cell size | same as transverse | $L_{\rm LOS}/N_z$ |
| Field access | `coeval.brightness_temp` | `lightcone.lightcones['brightness_temp']` |
| Neutral fraction | `coeval.xH_box` | `lightcone.lightcones['neutral_fraction']` |

### To improve for publication

- Use larger boxes (1 Gpc, 2 Mpc cells) as in Davies et al. (2025)
- Replace simplified noise with [21cmSense](https://github.com/rasg-affiliates/21cmSense)
- Enable `apply_rsds=True` in `generate_lightcone` for self-consistent RSDs
- Sweep over $\sigma_z$ and magnitude limits to optimise survey design

### References

- Davies, Mesinger & Murray (2025) — [arXiv:2504.17254](https://arxiv.org/abs/2504.17254)
- Gagnon-Hartman, Davies & Mesinger (2025) — [arXiv:2502.20447](https://arxiv.org/abs/2502.20447)
- La Plante et al. (2023) — [arXiv:2205.09770](https://arxiv.org/abs/2205.09770)
- Murray, Greig & Mesinger (2020) — [JOSS 5(54)](https://doi.org/10.21105/joss.02582)
